STAGE 4.1
# CORRECTED FAIR COMPARISON PROTOCOL

In [3]:
# ============================================================
# CISLR — STAGE 4.1
# Lightweight Model Search — Dataset Constraint Analysis
#
# PURPOSE:
#   1. Verify the official CISLR v1.5-a prototype/test structure
#   2. Prove whether an independent closed-set validation split
#      is possible using only the non-test pool
#   3. Freeze a fair comparison protocol for:
#         ResNet18
#         MobileNetV3
#         EfficientNet-B0
#
# IMPORTANT:
#   - NO model training
#   - NO official test evaluation
#   - NO new train/validation split
#   - NO modification of raw dataset files
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
from datetime import datetime

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT = Path("/home/dipshikha/Music/cao")

DATASET_CSV = ROOT / "dataset.csv"
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV = ROOT / "test.csv"

OUT_DIR = ROOT / "CISLR_RESEARCH" / "audit" / "stage4_1_constraints"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

dataset = pd.read_csv(DATASET_CSV)
prototype = pd.read_csv(PROTOTYPE_CSV)
test = pd.read_csv(TEST_CSV)

print("=" * 80)
print("CISLR — STAGE 4.1 DATASET CONSTRAINT ANALYSIS")
print("=" * 80)

print("\nDataset:")
print(f"  Rows       : {len(dataset):,}")
print(f"  Unique UIDs: {dataset['uid'].nunique():,}")

print("\nPrototype:")
print(f"  Rows       : {len(prototype):,}")
print(f"  Unique UIDs: {prototype['uid'].nunique():,}")

print("\nOfficial Test:")
print(f"  Rows       : {len(test):,}")
print(f"  Unique UIDs: {test['uid'].nunique():,}")

# ------------------------------------------------------------
# 3. BASIC UID INTEGRITY
# ------------------------------------------------------------

dataset_uids = set(dataset["uid"].dropna().astype(str))
prototype_uids = set(prototype["uid"].dropna().astype(str))
test_uids = set(test["uid"].dropna().astype(str))

prototype_test_overlap = prototype_uids & test_uids

print("\n" + "-" * 80)
print("UID PARTITION CHECK")
print("-" * 80)

print(f"Prototype/Test UID overlap : {len(prototype_test_overlap)}")

assert len(prototype_test_overlap) == 0, \
    "ERROR: Prototype and official test UIDs overlap."

print("PASS: Prototype and official test sets are UID-disjoint.")

# ------------------------------------------------------------
# 4. IDENTIFY VALID LABELLED PROTOTYPES
# ------------------------------------------------------------

# CISLR has one malformed/missing-label UID.
# We exclude it only from supervised analysis.
# Raw files remain untouched.

VALID_LABEL_COL = "gloss"

valid_prototype = prototype[
    prototype[VALID_LABEL_COL].notna()
].copy()

valid_prototype["uid"] = valid_prototype["uid"].astype(str)
valid_prototype["gloss"] = valid_prototype["gloss"].astype(str)

missing_label_prototype = prototype[
    prototype[VALID_LABEL_COL].isna()
].copy()

print("\n" + "-" * 80)
print("LABEL VALIDITY")
print("-" * 80)

print(f"Valid prototype videos       : {len(valid_prototype):,}")
print(f"Missing-label prototype rows : {len(missing_label_prototype):,}")

if len(missing_label_prototype) > 0:
    print("\nMissing-label UID(s):")
    for uid in missing_label_prototype["uid"].astype(str):
        print(f"  {uid}")

# ------------------------------------------------------------
# 5. VERIFY ONE PROTOTYPE VIDEO PER CLASS
# ------------------------------------------------------------

class_counts = valid_prototype.groupby("gloss")["uid"].nunique()

classes_with_one_video = int((class_counts == 1).sum())
classes_with_multiple_videos = int((class_counts > 1).sum())

print("\n" + "-" * 80)
print("PROTOTYPE CLASS SUPPORT")
print("-" * 80)

print(f"Unique valid prototype classes : {len(class_counts):,}")
print(f"Classes with exactly 1 video   : {classes_with_one_video:,}")
print(f"Classes with >1 prototype      : {classes_with_multiple_videos:,}")
print(f"Maximum prototype videos/class : {class_counts.max()}")

one_video_per_class = (
    len(class_counts) == len(valid_prototype)
    and class_counts.max() == 1
)

print(
    "\nOne-video-per-class prototype structure:",
    "YES" if one_video_per_class else "NO"
)

# ------------------------------------------------------------
# 6. VERIFY OFFICIAL TEST CLASS STRUCTURE
# ------------------------------------------------------------

valid_test = test[test[VALID_LABEL_COL].notna()].copy()

valid_test["uid"] = valid_test["uid"].astype(str)
valid_test["gloss"] = valid_test["gloss"].astype(str)

test_class_counts = valid_test.groupby("gloss")["uid"].nunique()

print("\n" + "-" * 80)
print("OFFICIAL TEST CLASS STRUCTURE")
print("-" * 80)

print(f"Valid official test videos : {len(valid_test):,}")
print(f"Official test classes      : {len(test_class_counts):,}")

# ------------------------------------------------------------
# 7. VERIFY PROTOTYPE -> TEST CLASS RELATIONSHIP
# ------------------------------------------------------------

prototype_classes = set(valid_prototype["gloss"])
test_classes = set(valid_test["gloss"])

prototype_only_classes = prototype_classes - test_classes
shared_classes = prototype_classes & test_classes
test_only_classes = test_classes - prototype_classes

print("\nPrototype/Test class relationship:")
print(f"  Prototype-only classes : {len(prototype_only_classes):,}")
print(f"  Shared classes         : {len(shared_classes):,}")
print(f"  Test-only classes      : {len(test_only_classes):,}")

# ------------------------------------------------------------
# 8. FORMAL PROOF OF VALIDATION CONSTRAINT
# ------------------------------------------------------------
#
# If every non-test prototype video belongs to a unique class,
# then ANY partition of those videos into train and validation
# necessarily produces:
#
#     TRAIN classes ∩ VALIDATION classes = ∅
#
# Therefore a conventional closed-set validation set cannot be
# constructed from this pool.
#
# This is a structural property, NOT a random-seed problem.
# ------------------------------------------------------------

uids = valid_prototype["uid"].tolist()

# Demonstrate the property using a deterministic hypothetical
# 90/10 partition. This is ONLY a mathematical demonstration;
# it is NOT saved as a training split.

rng = np.random.default_rng(2026)
shuffled = rng.permutation(uids)

n_train_demo = int(len(shuffled) * 0.90)

demo_train_uids = set(shuffled[:n_train_demo])
demo_val_uids = set(shuffled[n_train_demo:])

demo_train = valid_prototype[
    valid_prototype["uid"].isin(demo_train_uids)
]

demo_val = valid_prototype[
    valid_prototype["uid"].isin(demo_val_uids)
]

demo_train_classes = set(demo_train["gloss"])
demo_val_classes = set(demo_val["gloss"])

class_overlap = demo_train_classes & demo_val_classes

print("\n" + "-" * 80)
print("HYPOTHETICAL 90/10 CLOSED-SET VALIDATION CHECK")
print("-" * 80)

print(f"Hypothetical training videos   : {len(demo_train):,}")
print(f"Hypothetical validation videos : {len(demo_val):,}")
print(f"Training classes               : {len(demo_train_classes):,}")
print(f"Validation classes             : {len(demo_val_classes):,}")
print(f"Class overlap                  : {len(class_overlap):,}")

if one_video_per_class:
    assert len(class_overlap) == 0

print(
    "\nResult:",
    "CLOSED-SET VALIDATION IMPOSSIBLE"
    if one_video_per_class
    else "Further analysis required"
)

# ------------------------------------------------------------
# 9. CHECK WHETHER ANY NON-TEST CLASS HAS MULTIPLE VIDEOS
# ------------------------------------------------------------

# This is important:
# We check the complete dataset after removing official test UIDs.
# If repeated classes only have their additional examples in test,
# then there are still no duplicate-class videos available for
# independent supervised validation.

non_test_dataset = dataset[
    ~dataset["uid"].astype(str).isin(test_uids)
].copy()

non_test_dataset = non_test_dataset[
    non_test_dataset["gloss"].notna()
].copy()

non_test_dataset["uid"] = non_test_dataset["uid"].astype(str)
non_test_dataset["gloss"] = non_test_dataset["gloss"].astype(str)

non_test_class_counts = (
    non_test_dataset.groupby("gloss")["uid"].nunique()
)

classes_with_multiple_non_test_videos = (
    non_test_class_counts[non_test_class_counts > 1]
)

print("\n" + "-" * 80)
print("COMPLETE NON-TEST POOL CHECK")
print("-" * 80)

print(f"Valid non-test videos       : {len(non_test_dataset):,}")
print(f"Non-test classes            : {len(non_test_class_counts):,}")
print(
    f"Classes with >1 non-test video : "
    f"{len(classes_with_multiple_non_test_videos):,}"
)

if len(classes_with_multiple_non_test_videos) == 0:
    print(
        "\nCONFIRMED: There are no repeated-class videos "
        "available outside the official test set."
    )
else:
    print(
        "\nSome repeated-class non-test videos exist."
        " A constrained validation design may be possible."
    )

# ------------------------------------------------------------
# 10. FREEZE FAIR MODEL COMPARISON PROTOCOL
# ------------------------------------------------------------

protocol = {
    "stage": "4.1",
    "project": "CISLR v1.5-a",
    "purpose": "Lightweight model search",

    "dataset": {
        "dataset_rows": int(len(dataset)),
        "prototype_rows": int(len(prototype)),
        "test_rows": int(len(test)),
        "valid_prototype_videos": int(len(valid_prototype)),
        "valid_test_videos": int(len(valid_test)),
        "valid_prototype_classes": int(len(class_counts)),
        "one_prototype_video_per_class": bool(one_video_per_class),
        "prototype_test_uid_overlap": int(len(prototype_test_overlap)),
        "non_test_classes_with_multiple_videos":
            int(len(classes_with_multiple_non_test_videos))
    },

    "validation_constraint": {
        "independent_video_level_closed_set_validation": False,
        "reason":
            "The non-test prototype pool contains one video per class. "
            "Any video-level split therefore creates validation classes "
            "with no corresponding training example.",
        "official_test_used_for_model_selection": False,
        "new_random_split_created": False
    },

    "candidate_models": [
        "ResNet18",
        "MobileNetV3",
        "EfficientNet-B0"
    ],

    "input_protocol": {
        "image_size": "224x224",
        "channels": 3,
        "color_space": "RGB",
        "normalization": "ImageNet mean/std",
        "frames_per_video": 8,
        "sampling": "uniform temporal sampling"
    },

    "training_comparison": {
        "same_training_data": True,
        "same_input_pipeline": True,
        "same_augmentation_policy": True,
        "same_epoch_budget": True,
        "same_optimizer_policy": True,
        "same_batch_size": True,
        "same_seed_policy": True
    },

    "efficiency_metrics": [
        "parameter_count",
        "checkpoint_size_mb",
        "theoretical_flops",
        "batch1_gpu_latency_ms",
        "batch1_cpu_latency_ms",
        "gpu_memory_mb",
        "fps"
    ],

    "accuracy_evidence": {
        "independent_closed_set_validation": False,
        "training_diagnostics_allowed": True,
        "historical_posthoc_results_allowed_as_context": True,
        "official_test_reserved_for_final_evaluation": True
    },

    "selection_principle":
        "Select the backbone using controlled computational efficiency "
        "and available supervised evidence, without using the official "
        "test set for model selection."
}

# ------------------------------------------------------------
# 11. SAVE AUDIT REPORT
# ------------------------------------------------------------

report_rows = [
    ["dataset_rows", len(dataset)],
    ["prototype_rows", len(prototype)],
    ["test_rows", len(test)],
    ["valid_prototype_videos", len(valid_prototype)],
    ["valid_prototype_classes", len(class_counts)],
    ["one_video_per_prototype_class", one_video_per_class],
    ["prototype_test_uid_overlap", len(prototype_test_overlap)],
    ["valid_test_videos", len(valid_test)],
    ["valid_test_classes", len(test_class_counts)],
    ["non_test_classes_with_multiple_videos",
     len(classes_with_multiple_non_test_videos)],
    ["independent_closed_set_validation_possible", False],
    ["official_test_used_for_selection", False],
    ["new_random_validation_split_created", False],
]

report_df = pd.DataFrame(
    report_rows,
    columns=["metric", "value"]
)

report_path = OUT_DIR / "stage4_1_constraint_report.csv"
protocol_path = OUT_DIR / "stage4_1_protocol.json"

report_df.to_csv(report_path, index=False)

with open(protocol_path, "w") as f:
    json.dump(protocol, f, indent=2)

# ------------------------------------------------------------
# 12. HASH OUTPUTS FOR TRACEABILITY
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("\n" + "=" * 80)
print("STAGE 4.1 FINAL RESULT")
print("=" * 80)

print(f"""
Dataset integrity:
  Prototype/Test UID overlap        : {len(prototype_test_overlap)}
  Valid prototype videos            : {len(valid_prototype):,}
  Valid prototype classes            : {len(class_counts):,}
  One video per prototype class      : {one_video_per_class}
  Non-test classes with >1 video     : {len(classes_with_multiple_non_test_videos):,}

Validation:
  Independent closed-set validation  : IMPOSSIBLE
  New random split created           : NO
  Official test used for selection   : NO

Candidate backbones:
  1. ResNet18
  2. MobileNetV3
  3. EfficientNet-B0

Common input:
  224x224 RGB
  8 uniformly sampled frames
  ImageNet normalization

Efficiency metrics:
  Parameters
  Model size
  FLOPs
  CPU latency
  GPU latency
  GPU memory
  FPS

Artifacts:
  {report_path}
  {protocol_path}
""")

print("SHA256:")
print(f"  constraint_report.csv : {sha256_file(report_path)}")
print(f"  protocol.json         : {sha256_file(protocol_path)}")

print("\nStage 4.1 COMPLETE.")

CISLR — STAGE 4.1 DATASET CONSTRAINT ANALYSIS

Dataset:
  Rows       : 7,050
  Unique UIDs: 7,050

Prototype:
  Rows       : 4,765
  Unique UIDs: 4,765

Official Test:
  Rows       : 2,285
  Unique UIDs: 2,285

--------------------------------------------------------------------------------
UID PARTITION CHECK
--------------------------------------------------------------------------------
Prototype/Test UID overlap : 0
PASS: Prototype and official test sets are UID-disjoint.

--------------------------------------------------------------------------------
LABEL VALIDITY
--------------------------------------------------------------------------------
Valid prototype videos       : 4,764
Missing-label prototype rows : 1

Missing-label UID(s):
  R_GaklgD4bY=-

--------------------------------------------------------------------------------
PROTOTYPE CLASS SUPPORT
--------------------------------------------------------------------------------
Unique valid prototype classes : 4,764
Classe

 STAGE 4.2
# COMMON TRAINING PROTOCOL

In [9]:
# ============================================================
# CISLR — STAGE 4.2 UPDATED
# COMMON TRAINING PROTOCOL + TRAINING POOL RECONCILIATION
#
# PURPOSE:
#   Resolve the discrepancy between:
#       Historical Stage 3 pool : 4,417 videos / 35,336 frames
#       Current Stage 3 pool    : 4,764 videos / 38,112 frames
#
# PRINCIPLE:
#   Stage 4 must use ONE identical, reproducible training pool
#   for ResNet18, MobileNetV3-Large and EfficientNet-B0.
#
# IMPORTANT:
#   - No model training
#   - No official-test evaluation
#   - No new validation split
#   - Raw videos are never modified/deleted
#   - Existing Stage 3 artifacts are read-only
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import torch
import torchvision
import cv2
import sys
import platform
from datetime import datetime

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT = Path("/home/dipshikha/Music/cao")

DATASET_CSV = ROOT / "dataset.csv"
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV = ROOT / "test.csv"

STAGE3_DIR = ROOT / "CISLR_RESEARCH" / "processed"

STAGE3_TRAIN_META = (
    STAGE3_DIR / "stage3_train_frame_metadata.csv"
)

STAGE3_FRAME_META = (
    STAGE3_DIR / "stage3_frame_metadata.csv"
)

AUDIT_ROOT = (
    ROOT / "CISLR_RESEARCH" / "audit"
)

OUT_DIR = (
    AUDIT_ROOT / "stage4_2_protocol"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# New canonical manifest
CANONICAL_MANIFEST = (
    OUT_DIR / "stage4_2_canonical_training_manifest.csv"
)

RECONCILIATION_CSV = (
    OUT_DIR / "stage4_2_training_pool_reconciliation.csv"
)

PROTOCOL_JSON = (
    OUT_DIR / "stage4_2_common_training_protocol.json"
)

AUDIT_CSV = (
    OUT_DIR / "stage4_2_protocol_audit.csv"
)

# ------------------------------------------------------------
# 2. HISTORICAL EXPECTATIONS
# ------------------------------------------------------------

HISTORICAL_TRAIN_VIDEOS = 4417
HISTORICAL_TRAIN_FRAMES = 35336

CURRENT_EXPECTED_VALID_PROTOTYPE_VIDEOS = 4764
CURRENT_EXPECTED_VALID_PROTOTYPE_FRAMES = 38112

# Number of videos previously excluded
HISTORICAL_EXCLUDED_COUNT = (
    CURRENT_EXPECTED_VALID_PROTOTYPE_VIDEOS
    - HISTORICAL_TRAIN_VIDEOS
)

# ------------------------------------------------------------
# 3. LOAD OFFICIAL DATA
# ------------------------------------------------------------

dataset = pd.read_csv(DATASET_CSV)
prototype = pd.read_csv(PROTOTYPE_CSV)
test = pd.read_csv(TEST_CSV)

print("=" * 80)
print("CISLR — STAGE 4.2 UPDATED")
print("COMMON TRAINING PROTOCOL + TRAINING POOL RECONCILIATION")
print("=" * 80)

# ------------------------------------------------------------
# 4. UID INTEGRITY
# ------------------------------------------------------------

dataset_uids = set(
    dataset["uid"].dropna().astype(str)
)

prototype_uids = set(
    prototype["uid"].dropna().astype(str)
)

test_uids = set(
    test["uid"].dropna().astype(str)
)

prototype_test_overlap = (
    prototype_uids & test_uids
)

assert len(prototype_test_overlap) == 0, (
    "ERROR: Prototype/Test UID overlap detected."
)

print("\n" + "-" * 80)
print("OFFICIAL DATASET INTEGRITY")
print("-" * 80)

print(f"Dataset UIDs             : {len(dataset_uids):,}")
print(f"Prototype UIDs           : {len(prototype_uids):,}")
print(f"Official Test UIDs       : {len(test_uids):,}")
print(
    f"Prototype/Test overlap   : "
    f"{len(prototype_test_overlap)}"
)

# ------------------------------------------------------------
# 5. VALID PROTOTYPE POOL
# ------------------------------------------------------------

valid_prototype = prototype[
    prototype["gloss"].notna()
].copy()

valid_prototype["uid"] = (
    valid_prototype["uid"]
    .astype(str)
)

valid_prototype["gloss"] = (
    valid_prototype["gloss"]
    .astype(str)
)

# One video per class
prototype_class_counts = (
    valid_prototype
    .groupby("gloss")["uid"]
    .nunique()
)

assert (
    prototype_class_counts.max() == 1
), "ERROR: Prototype contains repeated class videos."

assert (
    len(valid_prototype)
    == len(prototype_class_counts)
), "ERROR: Prototype is not one-video-per-class."

print("\nValid prototype videos : "
      f"{len(valid_prototype):,}")

print("Valid prototype classes: "
      f"{len(prototype_class_counts):,}")

# ------------------------------------------------------------
# 6. LOAD CURRENT STAGE 3 TRAINING METADATA
# ------------------------------------------------------------

if not STAGE3_TRAIN_META.exists():
    raise FileNotFoundError(
        f"\nMissing Stage 3 training metadata:\n"
        f"{STAGE3_TRAIN_META}"
    )

train_meta = pd.read_csv(
    STAGE3_TRAIN_META
)

# Detect UID column
uid_column_candidates = [
    "uid",
    "video_id",
    "video_uid"
]

uid_col = None

for col in uid_column_candidates:
    if col in train_meta.columns:
        uid_col = col
        break

if uid_col is None:
    raise KeyError(
        "Could not identify video UID column in "
        "stage3_train_frame_metadata.csv"
    )

# Detect image path column
path_column_candidates = [
    "image_path",
    "frame_path",
    "path"
]

path_col = None

for col in path_column_candidates:
    if col in train_meta.columns:
        path_col = col
        break

if path_col is None:
    path_column_candidates = [
        "image_path",
        "frame_path",
        "video_path",
        "path",
        "file_path",
        "filepath",
        "image_file",
        "frame_file"
    ]

    path_col = None

    for col in path_column_candidates:
        if col in train_meta.columns:
            path_col = col
            break

    if path_col is None:
        # Fallback: detect any likely path-like column
        for col in train_meta.columns:
            name = str(col).lower()
            if (
                ("path" in name or "file" in name or "image" in name)
                and "uid" not in name
                and "video_id" not in name
            ):
                path_col = col
                break

    if path_col is None:
        raise KeyError(
            "Could not identify image/video path column in "
            "stage3_train_frame_metadata.csv. "
            f"Available columns: {list(train_meta.columns)}"
        )

train_meta[uid_col] = (
    train_meta[uid_col].astype(str)
)

current_train_uids = set(
    train_meta[uid_col]
)

print("\n" + "-" * 80)
print("CURRENT STAGE 3 TRAINING METADATA")
print("-" * 80)

print(
    f"Frame rows       : {len(train_meta):,}"
)

print(
    f"Unique videos    : {len(current_train_uids):,}"
)

# ------------------------------------------------------------
# 7. VERIFY CURRENT POOL IS NON-TEST
# ------------------------------------------------------------

current_test_overlap = (
    current_train_uids & test_uids
)

assert len(current_test_overlap) == 0, (
    "ERROR: Current training metadata contains "
    "official test UIDs."
)

print(
    f"Training/Test overlap : "
    f"{len(current_test_overlap)}"
)

# ------------------------------------------------------------
# 8. VERIFY CURRENT POOL IS PROTOTYPE-BASED
# ------------------------------------------------------------

outside_prototype = (
    current_train_uids - prototype_uids
)

print(
    f"Training UIDs outside prototype : "
    f"{len(outside_prototype):,}"
)

assert len(outside_prototype) == 0, (
    "ERROR: Current training pool contains UIDs "
    "outside prototype.csv."
)

# ------------------------------------------------------------
# 9. CURRENT VS HISTORICAL DISCREPANCY
# ------------------------------------------------------------

current_video_count = len(current_train_uids)
current_frame_count = len(train_meta)

video_difference = (
    current_video_count
    - HISTORICAL_TRAIN_VIDEOS
)

frame_difference = (
    current_frame_count
    - HISTORICAL_TRAIN_FRAMES
)

print("\n" + "-" * 80)
print("TRAINING POOL RECONCILIATION")
print("-" * 80)

print(
    f"Historical Stage 3 videos : "
    f"{HISTORICAL_TRAIN_VIDEOS:,}"
)

print(
    f"Current Stage 3 videos    : "
    f"{current_video_count:,}"
)

print(
    f"Video difference           : "
    f"{video_difference:+,}"
)

print(
    f"Historical Stage 3 frames : "
    f"{HISTORICAL_TRAIN_FRAMES:,}"
)

print(
    f"Current Stage 3 frames    : "
    f"{current_frame_count:,}"
)

print(
    f"Frame difference           : "
    f"{frame_difference:+,}"
)

# ------------------------------------------------------------
# 10. IDENTIFY THE 347 DISCREPANT VIDEOS
# ------------------------------------------------------------

historical_difference_uids = (
    current_train_uids
    - set()
)

# The historical 4,417 UID list may exist inside prior
# audit artifacts. Search known CSV files recursively.
#
# We only READ CSV files. Nothing is modified.

historical_uid_candidates = []

for csv_path in AUDIT_ROOT.rglob("*.csv"):

    try:
        df_tmp = pd.read_csv(
            csv_path,
            nrows=5
        )
    except Exception:
        continue

    cols_lower = {
        str(c).lower(): c
        for c in df_tmp.columns
    }

    possible_uid_col = None

    for candidate in [
        "uid",
        "video_id",
        "video_uid"
    ]:
        if candidate in cols_lower:
            possible_uid_col = cols_lower[candidate]
            break

    if possible_uid_col is not None:

        try:
            full_tmp = pd.read_csv(
                csv_path,
                usecols=[possible_uid_col]
            )

            vals = set(
                full_tmp[possible_uid_col]
                .dropna()
                .astype(str)
            )

            # Look specifically for the historical 4,417 pool.
            if len(vals) == HISTORICAL_TRAIN_VIDEOS:

                overlap = (
                    vals & current_train_uids
                )

                if len(overlap) >= 4000:

                    historical_uid_candidates.append(
                        (
                            csv_path,
                            vals,
                            len(overlap)
                        )
                    )

        except Exception:
            pass

print("\nHistorical UID manifests found:")
print(
    f"  Candidate manifests : "
    f"{len(historical_uid_candidates)}"
)

# ------------------------------------------------------------
# 11. SELECT HISTORICAL UID MANIFEST IF UNAMBIGUOUS
# ------------------------------------------------------------

historical_uids = None
historical_manifest_path = None

if len(historical_uid_candidates) > 0:

    # Prefer exact 4,417 overlap.
    exact_candidates = [
        x for x in historical_uid_candidates
        if x[2] == HISTORICAL_TRAIN_VIDEOS
    ]

    if len(exact_candidates) == 1:

        historical_manifest_path = (
            exact_candidates[0][0]
        )

        historical_uids = (
            exact_candidates[0][1]
        )

        print(
            "\nHistorical training manifest identified:"
        )

        print(
            f"  {historical_manifest_path}"
        )

    elif len(exact_candidates) > 1:

        # If multiple identical candidates exist,
        # compare their UID sets.
        unique_sets = []

        for _, uid_set, _ in exact_candidates:

            if uid_set not in unique_sets:
                unique_sets.append(uid_set)

        if len(unique_sets) == 1:

            historical_manifest_path = (
                exact_candidates[0][0]
            )

            historical_uids = unique_sets[0]

            print(
                "\nMultiple identical historical manifests "
                "found. UID set is identical."
            )

            print(
                f"Using: {historical_manifest_path}"
            )

        else:

            print(
                "\nWARNING: Multiple different historical "
                "UID manifests found."
            )

    else:

        print(
            "\nWARNING: Candidate historical manifests "
            "found, but none has exact 4,417 UID overlap."
        )

# ------------------------------------------------------------
# 12. IDENTIFY DISCREPANT VIDEOS
# ------------------------------------------------------------

if historical_uids is not None:

    added_since_historical = (
        current_train_uids
        - historical_uids
    )

    removed_since_historical = (
        historical_uids
        - current_train_uids
    )

else:

    added_since_historical = set()
    removed_since_historical = set()

print("\n" + "-" * 80)
print("HISTORICAL ↔ CURRENT DIFFERENCE")
print("-" * 80)

if historical_uids is not None:

    print(
        f"Historical pool : {len(historical_uids):,}"
    )

    print(
        f"Current pool    : {len(current_train_uids):,}"
    )

    print(
        f"Added videos    : "
        f"{len(added_since_historical):,}"
    )

    print(
        f"Removed videos  : "
        f"{len(removed_since_historical):,}"
    )

    if len(added_since_historical) == HISTORICAL_EXCLUDED_COUNT:

        print(
            "\nCONFIRMED: Current metadata re-includes "
            "exactly the 347 videos previously excluded."
        )

else:

    print(
        "Historical 4,417-video UID manifest could not "
        "be located automatically."
    )

# ------------------------------------------------------------
# 13. DECODABILITY AUDIT OF DISCREPANT VIDEOS
# ------------------------------------------------------------
#
# We now determine whether the 347 videos are genuinely
# corrupted or merely excluded by an older pipeline.
#
# We DO NOT modify/delete them.
#
# We test a deterministic set of frames:
#     first frame
#     middle frame
#     last reported frame
#
# A video is considered "decode-valid" if all three checks
# succeed.
# ------------------------------------------------------------

# Need video path column from dataset.csv.
video_path_col_candidates = [
    "video_path",
    "path",
    "video",
    "filepath",
    "file_path"
]

video_path_col = None

for col in video_path_col_candidates:

    if col in dataset.columns:
        video_path_col = col
        break

if video_path_col is None:

    # Sometimes dataset stores filename rather than full path.
    filename_candidates = [
        "filename",
        "file_name",
        "video_name"
    ]

    for col in filename_candidates:

        if col in dataset.columns:
            video_path_col = col
            break

# ------------------------------------------------------------
# 14. FIND VIDEO FILE BY UID IF NECESSARY
# ------------------------------------------------------------

VIDEO_ROOT = (
    ROOT / "CISLR_v1.5-a_videos"
)

def resolve_video_path(uid):

    # 1. Try exact UID filename.
    candidates = [
        VIDEO_ROOT / f"{uid}.mp4",
        VIDEO_ROOT / uid,
    ]

    for p in candidates:
        if p.exists():
            return p

    # 2. Search dataset metadata.
    if video_path_col is not None:

        rows = dataset[
            dataset["uid"]
            .astype(str)
            .eq(str(uid))
        ]

        if len(rows) > 0:

            raw = rows.iloc[0][video_path_col]

            if pd.notna(raw):

                raw = str(raw)

                p = Path(raw)

                if p.exists():
                    return p

                p2 = VIDEO_ROOT / p.name

                if p2.exists():
                    return p2

    # 3. Recursive filename search.
    matches = list(
        VIDEO_ROOT.rglob(f"{uid}.mp4")
    )

    if len(matches) > 0:
        return matches[0]

    return None

# ------------------------------------------------------------
# 15. AUDIT ONLY THE HISTORICAL 347 IF IDENTIFIED
# ------------------------------------------------------------

decode_results = []

if len(added_since_historical) > 0:

    print("\n" + "-" * 80)
    print("DECODABILITY AUDIT OF DISCREPANT VIDEOS")
    print("-" * 80)

    for i, uid in enumerate(
        sorted(added_since_historical),
        start=1
    ):

        video_path = resolve_video_path(uid)

        result = {
            "uid": uid,
            "video_path": (
                str(video_path)
                if video_path is not None
                else None
            ),
            "file_exists": False,
            "opened": False,
            "frame_count": None,
            "fps": None,
            "first_frame_ok": False,
            "middle_frame_ok": False,
            "last_frame_ok": False,
            "decode_valid": False,
            "error": None
        }

        try:

            if video_path is None:

                result["error"] = (
                    "Video file could not be resolved."
                )

            else:

                result["file_exists"] = (
                    video_path.exists()
                )

                cap = cv2.VideoCapture(
                    str(video_path)
                )

                if not cap.isOpened():

                    result["error"] = (
                        "OpenCV could not open video."
                    )

                else:

                    result["opened"] = True

                    frame_count = int(
                        cap.get(
                            cv2.CAP_PROP_FRAME_COUNT
                        )
                    )

                    fps = float(
                        cap.get(
                            cv2.CAP_PROP_FPS
                        )
                    )

                    result["frame_count"] = (
                        frame_count
                    )

                    result["fps"] = fps

                    if frame_count > 0:

                        # First
                        cap.set(
                            cv2.CAP_PROP_POS_FRAMES,
                            0
                        )

                        ok, frame = cap.read()

                        result["first_frame_ok"] = (
                            bool(ok)
                            and frame is not None
                        )

                        # Middle
                        middle_idx = (
                            frame_count // 2
                        )

                        cap.set(
                            cv2.CAP_PROP_POS_FRAMES,
                            middle_idx
                        )

                        ok, frame = cap.read()

                        result["middle_frame_ok"] = (
                            bool(ok)
                            and frame is not None
                        )

                        # Last reported frame
                        last_idx = max(
                            0,
                            frame_count - 1
                        )

                        cap.set(
                            cv2.CAP_PROP_POS_FRAMES,
                            last_idx
                        )

                        ok, frame = cap.read()

                        result["last_frame_ok"] = (
                            bool(ok)
                            and frame is not None
                        )

                    result["decode_valid"] = all([
                        result["first_frame_ok"],
                        result["middle_frame_ok"],
                        result["last_frame_ok"]
                    ])

                    cap.release()

        except Exception as e:

            result["error"] = str(e)

        decode_results.append(result)

        if i % 25 == 0 or i == len(
            added_since_historical
        ):

            print(
                f"Checked {i}/"
                f"{len(added_since_historical)}"
            )

else:

    print(
        "\nNo 347-video discrepancy could be "
        "identified automatically."
    )

decode_df = pd.DataFrame(
    decode_results
)

# ------------------------------------------------------------
# 16. DECIDE CANONICAL TRAINING POOL
# ------------------------------------------------------------

if (
    historical_uids is not None
    and len(added_since_historical)
    == HISTORICAL_EXCLUDED_COUNT
):

    if (
        len(decode_df) > 0
        and decode_df["decode_valid"].all()
    ):

        # All previously excluded videos are now demonstrably
        # decodable. Therefore they should NOT be excluded merely
        # because an older run excluded them.
        canonical_uids = set(
            current_train_uids
        )

        pool_decision = (
            "USE_CURRENT_4764_POOL"
        )

        decision_reason = (
            "The 347 videos previously absent from the historical "
            "4,417-video pool are now demonstrably decodable under "
            "the Stage 4 deterministic audit. Therefore the full "
            "4,764-video prototype pool is retained."
        )

    else:

        # At least one discrepancy video failed the audit.
        bad_uids = set(
            decode_df.loc[
                ~decode_df["decode_valid"],
                "uid"
            ]
        )

        canonical_uids = (
            current_train_uids
            - bad_uids
        )

        pool_decision = (
            "EXCLUDE_DECODE_INVALID_VIDEOS"
        )

        decision_reason = (
            "One or more previously excluded videos failed the "
            "deterministic decode audit. Only demonstrably invalid "
            "videos are excluded; raw files remain untouched."
        )

else:

    # No reliable historical UID manifest.
    # Do not guess which videos were previously excluded.
    #
    # Current metadata is internally consistent:
    # 4,764 videos × 8 frames = 38,112 frames.
    #
    # Therefore use current pool unless actual corruption is
    # independently demonstrated.
    canonical_uids = set(
        current_train_uids
    )

    pool_decision = (
        "USE_CURRENT_POOL_NO_HISTORICAL_UID_MANIFEST"
    )

    decision_reason = (
        "The historical 4,417-video UID manifest could not be "
        "unambiguously recovered. The current Stage 3 metadata "
        "is internally consistent at 4,764 videos × 8 frames and "
        "contains no official-test UIDs. Therefore no videos are "
        "excluded based solely on historical provenance."
    )

# ------------------------------------------------------------
# 17. BUILD CANONICAL FRAME MANIFEST
# ------------------------------------------------------------

canonical_meta = train_meta[
    train_meta[uid_col].isin(canonical_uids)
].copy()

canonical_meta = canonical_meta.sort_values(
    by=[uid_col]
).reset_index(drop=True)

# Verify exactly 8 frames/video.
frames_per_video = (
    canonical_meta
    .groupby(uid_col)
    .size()
)

bad_frame_counts = (
    frames_per_video[
        frames_per_video != 8
    ]
)

if len(bad_frame_counts) > 0:

    print("\nWARNING:")
    print(
        f"{len(bad_frame_counts)} videos do not have "
        f"exactly 8 metadata rows."
    )

    # Do NOT silently fix them.
    raise RuntimeError(
        "Canonical training manifest contains videos "
        "with frame counts != 8. Resolve before Stage 4.3."
    )

# ------------------------------------------------------------
# 18. FINAL CANONICAL POOL INTEGRITY
# ------------------------------------------------------------

canonical_test_overlap = (
    canonical_uids & test_uids
)

canonical_prototype_outside = (
    canonical_uids - prototype_uids
)

assert len(canonical_test_overlap) == 0
assert len(canonical_prototype_outside) == 0

canonical_video_count = len(
    canonical_uids
)

canonical_frame_count = len(
    canonical_meta
)

assert (
    canonical_frame_count
    == canonical_video_count * 8
)

# ------------------------------------------------------------
# 19. SAVE CANONICAL MANIFEST
# ------------------------------------------------------------

canonical_meta.to_csv(
    CANONICAL_MANIFEST,
    index=False
)

# ------------------------------------------------------------
# 20. RECONCILIATION TABLE
# ------------------------------------------------------------

reconciliation_rows = [
    {
        "pool": "historical_stage3",
        "videos": HISTORICAL_TRAIN_VIDEOS,
        "frames": HISTORICAL_TRAIN_FRAMES,
        "status": "historical_reference"
    },
    {
        "pool": "current_stage3",
        "videos": current_video_count,
        "frames": current_frame_count,
        "status": "current_metadata"
    },
    {
        "pool": "canonical_stage4",
        "videos": canonical_video_count,
        "frames": canonical_frame_count,
        "status": pool_decision
    }
]

reconciliation_df = pd.DataFrame(
    reconciliation_rows
)

reconciliation_df.to_csv(
    RECONCILIATION_CSV,
    index=False
)

# ------------------------------------------------------------
# 21. COMMON TRAINING PROTOCOL
# ------------------------------------------------------------

candidate_models = {
    "ResNet18": {
        "torchvision_name": "resnet18",
        "family": "ResNet",
        "variant": "18",
        "pretrained": True
    },

    "MobileNetV3-Large": {
        "torchvision_name": "mobilenet_v3_large",
        "family": "MobileNetV3",
        "variant": "Large",
        "pretrained": True
    },

    "EfficientNet-B0": {
        "torchvision_name": "efficientnet_b0",
        "family": "EfficientNet",
        "variant": "B0",
        "pretrained": True
    }
}

input_protocol = {
    "image_size": [224, 224],
    "channels": 3,
    "color_space": "RGB",
    "frames_per_video": 8,
    "sampling": "uniform",
    "normalization": {
        "type": "ImageNet",
        "mean": [
            0.485,
            0.456,
            0.406
        ],
        "std": [
            0.229,
            0.224,
            0.225
        ]
    }
}

training_protocol = {
    "epochs": 10,

    "batch_size": 16,

    "optimizer": {
        "name": "AdamW",
        "learning_rate": 1e-4,
        "weight_decay": 1e-4
    },

    "loss": "CrossEntropyLoss",

    "mixed_precision": {
        "enabled": True
    },

    "seed": 42,

    "num_workers": 4,

    "pin_memory": True,

    "persistent_workers": True
}

augmentation_protocol = {
    "training": [
        "RandomResizedCrop(224, scale=(0.8, 1.0))",
        "RandomHorizontalFlip(p=0.5)"
    ],

    "evaluation": [
        "Resize(256)",
        "CenterCrop(224)"
    ]
}

benchmark_protocol = {
    "batch_size": 1,
    "warmup_iterations": 50,
    "measurement_iterations": 200,
    "metrics": [
        "parameter_count",
        "trainable_parameter_count",
        "checkpoint_size_mb",
        "theoretical_flops",
        "gpu_latency_ms",
        "gpu_fps",
        "gpu_peak_memory_mb",
        "cpu_latency_ms",
        "cpu_fps"
    ]
}

# ------------------------------------------------------------
# 22. HARDWARE
# ------------------------------------------------------------

hardware = {
    "platform": platform.platform(),
    "python": sys.version,
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda
}

if torch.cuda.is_available():

    hardware["gpu"] = (
        torch.cuda.get_device_name(0)
    )

    hardware["gpu_total_memory_gb"] = round(
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3),
        3
    )

else:

    hardware["gpu"] = None
    hardware["gpu_total_memory_gb"] = None

# ------------------------------------------------------------
# 23. FINAL PROTOCOL OBJECT
# ------------------------------------------------------------

protocol = {

    "stage": "4.2",

    "title":
        "Common Training Protocol + Training Pool Reconciliation",

    "created_at":
        datetime.now().isoformat(),

    "training_pool": {

        "historical_videos":
            HISTORICAL_TRAIN_VIDEOS,

        "historical_frames":
            HISTORICAL_TRAIN_FRAMES,

        "current_videos":
            current_video_count,

        "current_frames":
            current_frame_count,

        "canonical_videos":
            canonical_video_count,

        "canonical_frames":
            canonical_frame_count,

        "decision":
            pool_decision,

        "reason":
            decision_reason,

        "canonical_manifest":
            str(CANONICAL_MANIFEST)
    },

    "dataset_protection": {

        "official_test_used":
            False,

        "official_test_used_for_selection":
            False,

        "raw_videos_modified":
            False,

        "raw_dataset_modified":
            False,

        "new_validation_split":
            False
    },

    "candidate_models":
        candidate_models,

    "input_protocol":
        input_protocol,

    "training_protocol":
        training_protocol,

    "augmentation_protocol":
        augmentation_protocol,

    "benchmark_protocol":
        benchmark_protocol,

    "hardware":
        hardware,

    "validation_constraint": {

        "independent_closed_set_validation":
            False,

        "reason":
            "The non-test prototype pool contains exactly one "
            "video per class."
    },

    "fairness_constraints": {

        "same_canonical_video_pool":
            True,

        "same_frames_per_video":
            True,

        "same_input_resolution":
            True,

        "same_normalization":
            True,

        "same_augmentation":
            True,

        "same_batch_size":
            True,

        "same_optimizer":
            True,

        "same_learning_rate":
            True,

        "same_weight_decay":
            True,

        "same_epoch_budget":
            True,

        "same_seed":
            True,

        "same_precision":
            True,

        "architecture_specific_tuning":
            False
    }
}

# ------------------------------------------------------------
# 24. AUDIT TABLE
# ------------------------------------------------------------

audit_rows = [

    ["Prototype/Test UID overlap",
     len(prototype_test_overlap),
     "PASS"],

    ["Current training/Test overlap",
     len(current_test_overlap),
     "PASS"],

    ["Current training UIDs outside prototype",
     len(outside_prototype),
     "PASS"],

    ["Historical training videos",
     HISTORICAL_TRAIN_VIDEOS,
     "REFERENCE"],

    ["Current training videos",
     current_video_count,
     "OBSERVED"],

    ["Current training frames",
     current_frame_count,
     "OBSERVED"],

    ["Historical/current video difference",
     video_difference,
     "RECONCILED"],

    ["Historical/current frame difference",
     frame_difference,
     "RECONCILED"],

    ["Canonical Stage 4 videos",
     canonical_video_count,
     "FROZEN"],

    ["Canonical Stage 4 frames",
     canonical_frame_count,
     "FROZEN"],

    ["Canonical frames/video",
     8,
     "PASS"],

    ["Canonical/Test overlap",
     len(canonical_test_overlap),
     "PASS"],

    ["Canonical/Prototype outside",
     len(canonical_prototype_outside),
     "PASS"],

    ["Independent closed-set validation",
     "IMPOSSIBLE",
     "DATASET CONSTRAINT"],

    ["Official test used",
     False,
     "PASS"],

    ["New validation split",
     False,
     "PASS"],

    ["Raw data modified",
     False,
     "PASS"]
]

audit_df = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "value",
        "status"
    ]
)

audit_df.to_csv(
    AUDIT_CSV,
    index=False
)

# ------------------------------------------------------------
# 25. SAVE PROTOCOL
# ------------------------------------------------------------

with open(
    PROTOCOL_JSON,
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 26. HASH FUNCTION
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()

# ------------------------------------------------------------
# 27. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STAGE 4.2 — FINAL RECONCILIATION")
print("=" * 80)

print(
    f"""
Historical Stage 3 pool:
  Videos : {HISTORICAL_TRAIN_VIDEOS:,}
  Frames : {HISTORICAL_TRAIN_FRAMES:,}

Current Stage 3 pool:
  Videos : {current_video_count:,}
  Frames : {current_frame_count:,}

Canonical Stage 4 pool:
  Videos : {canonical_video_count:,}
  Frames : {canonical_frame_count:,}

Decision:
  {pool_decision}

Reason:
  {decision_reason}
"""
)

print("-" * 80)
print("CANONICAL TRAINING POOL")
print("-" * 80)

print(
    f"Videos       : {canonical_video_count:,}"
)

print(
    f"Frames       : {canonical_frame_count:,}"
)

print(
    f"Frames/video : "
    f"{canonical_frame_count / canonical_video_count:.1f}"
)

print(
    f"Test overlap : "
    f"{len(canonical_test_overlap)}"
)

print(
    f"Prototype outside : "
    f"{len(canonical_prototype_outside)}"
)

print("\nCandidate backbones:")
print("  1. ResNet18")
print("  2. MobileNetV3-Large")
print("  3. EfficientNet-B0")

print("\nCommon training:")
print("  Input       : 224x224 RGB")
print("  Frames      : 8/video")
print("  Sampling    : Uniform")
print("  Normalization : ImageNet")
print("  Batch size  : 16")
print("  Epochs      : 10")
print("  Optimizer   : AdamW")
print("  LR          : 1e-4")
print("  WD          : 1e-4")
print("  Seed        : 42")
print("  AMP         : Enabled")

print("\nDataset protection:")
print("  Official test used : NO")
print("  New validation     : NO")
print("  Raw data modified  : NO")

print("\nArtifacts:")
print(f"  Canonical manifest : {CANONICAL_MANIFEST}")
print(f"  Reconciliation     : {RECONCILIATION_CSV}")
print(f"  Protocol           : {PROTOCOL_JSON}")
print(f"  Audit              : {AUDIT_CSV}")

print("\nSHA256:")
print(
    "  canonical_manifest : "
    f"{sha256_file(CANONICAL_MANIFEST)}"
)

print(
    "  reconciliation     : "
    f"{sha256_file(RECONCILIATION_CSV)}"
)

print(
    "  protocol           : "
    f"{sha256_file(PROTOCOL_JSON)}"
)

print(
    "  audit              : "
    f"{sha256_file(AUDIT_CSV)}"
)

print("\n" + "=" * 80)
print("STAGE 4.2 UPDATED — COMPLETE")
print("=" * 80)

CISLR — STAGE 4.2 UPDATED
COMMON TRAINING PROTOCOL + TRAINING POOL RECONCILIATION

--------------------------------------------------------------------------------
OFFICIAL DATASET INTEGRITY
--------------------------------------------------------------------------------
Dataset UIDs             : 7,050
Prototype UIDs           : 4,765
Official Test UIDs       : 2,285
Prototype/Test overlap   : 0

Valid prototype videos : 4,764
Valid prototype classes: 4,764

--------------------------------------------------------------------------------
CURRENT STAGE 3 TRAINING METADATA
--------------------------------------------------------------------------------
Frame rows       : 38,112
Unique videos    : 4,764
Training/Test overlap : 0
Training UIDs outside prototype : 0

--------------------------------------------------------------------------------
TRAINING POOL RECONCILIATION
--------------------------------------------------------------------------------
Historical Stage 3 videos : 4,417
C

STAGE 4.3
# COMPUTATIONAL BENCHMARKING

In [12]:
# =============================================================================
# CISLR — STAGE 4.3
# COMPUTATIONAL BENCHMARKING
# =============================================================================
#
# PURPOSE
# -------
# Fair computational comparison of:
#   1. ResNet18
#   2. MobileNetV3-Large
#   3. EfficientNet-B0
#
# IMPORTANT
# ---------
# - NO TRAINING
# - NO OFFICIAL TEST DATA
# - NO NEW DATA SPLIT
# - NO MODEL SELECTION USING OFFICIAL TEST
#
# Benchmark conditions:
#   Input             : 1 x 3 x 224 x 224
#   Classifier head   : 4764 classes
#   Precision         : FP32
#   Batch size        : 1
#   CPU warmup        : 50
#   CPU measurements  : 200
#   GPU warmup        : 50
#   GPU measurements  : 200
#   CPU threads       : 1
#
# FLOPs:
#   Analytical Conv2D + Linear count
#   1 multiply-add = 2 FLOPs
#
# Outputs:
#   CISLR_RESEARCH/audit/stage4_3_benchmark/
#       stage4_3_computational_benchmark.csv
#       stage4_3_computational_benchmark.json
#       stage4_3_protocol.json
#
# =============================================================================

import os
import io
import json
import time
import hashlib
import platform
from pathlib import Path

import torch
import torch.nn as nn
import torchvision
from torchvision import models
import pandas as pd


# =============================================================================
# 0. CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/home/dipshikha/Music/cao")

OUTPUT_DIR = (
    PROJECT_ROOT
    / "CISLR_RESEARCH"
    / "audit"
    / "stage4_3_benchmark"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 4764
IMAGE_SIZE = 224
BATCH_SIZE = 1

WARMUP_ITERS = 50
MEASURE_ITERS = 200

CPU_THREADS = 1

SEED = 42

torch.manual_seed(SEED)

print("=" * 80)
print("CISLR — STAGE 4.3 COMPUTATIONAL BENCHMARKING")
print("=" * 80)


# =============================================================================
# 1. ENVIRONMENT / HARDWARE
# =============================================================================

print("\n[1/8] Environment")

print(f"PyTorch       : {torch.__version__}")
print(f"Torchvision   : {torchvision.__version__}")
print(f"Python        : {platform.python_version()}")
print(f"Platform      : {platform.platform()}")

CUDA_AVAILABLE = torch.cuda.is_available()

print(f"CUDA available: {CUDA_AVAILABLE}")

if CUDA_AVAILABLE:
    GPU_NAME = torch.cuda.get_device_name(0)
    CUDA_VERSION = torch.version.cuda
    print(f"GPU           : {GPU_NAME}")
    print(f"CUDA          : {CUDA_VERSION}")
else:
    GPU_NAME = None
    CUDA_VERSION = None
    print("GPU           : None")

print(f"CPU threads   : {CPU_THREADS}")


# =============================================================================
# 2. VERIFY STAGE 4.2 CANONICAL POOL
# =============================================================================

print("\n[2/8] Verifying Stage 4.2 canonical training pool")

STAGE4_2_MANIFEST = (
    PROJECT_ROOT
    / "CISLR_RESEARCH"
    / "audit"
    / "stage4_2_protocol"
    / "stage4_2_canonical_training_manifest.csv"
)

if not STAGE4_2_MANIFEST.exists():
    raise FileNotFoundError(
        f"Stage 4.2 canonical manifest not found:\n{STAGE4_2_MANIFEST}"
    )

manifest = pd.read_csv(STAGE4_2_MANIFEST)

print(f"Manifest rows : {len(manifest):,}")

if "video_id" in manifest.columns:
    n_videos = manifest["video_id"].nunique()
else:
    video_id_col = next(
        (col for col in ["video_id", "video_uid", "uid"] if col in manifest.columns),
        None,
    )

    if video_id_col is None:
        raise RuntimeError(
            "Stage 4.2 manifest does not contain any usable video identifier "
            "column ('video_id', 'video_uid', or 'uid')."
        )

    n_videos = manifest[video_id_col].nunique()

print(f"Videos        : {n_videos:,}")

if n_videos != 4764:
    raise RuntimeError(
        f"Expected 4764 canonical videos, found {n_videos}."
    )

print("✓ Stage 4.2 canonical pool verified: 4764 videos")


# =============================================================================
# 3. MODEL BUILDERS
# =============================================================================

print("\n[3/8] Building candidate architectures")

def build_resnet18():
    """
    ResNet18 with ImageNet-pretrained backbone and
    CISLR 4764-class classifier head.
    """
    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT
    )

    in_features = model.fc.in_features

    model.fc = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    return model


def build_mobilenet_v3_large():
    """
    MobileNetV3-Large with ImageNet-pretrained backbone and
    CISLR 4764-class classifier head.
    """
    model = models.mobilenet_v3_large(
        weights=models.MobileNet_V3_Large_Weights.DEFAULT
    )

    in_features = model.classifier[-1].in_features

    model.classifier[-1] = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    return model


def build_efficientnet_b0():
    """
    EfficientNet-B0 with ImageNet-pretrained backbone and
    CISLR 4764-class classifier head.
    """
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.DEFAULT
    )

    in_features = model.classifier[-1].in_features

    model.classifier[-1] = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    return model


MODEL_BUILDERS = {
    "ResNet18": build_resnet18,
    "MobileNetV3-Large": build_mobilenet_v3_large,
    "EfficientNet-B0": build_efficientnet_b0,
}


# =============================================================================
# 4. ANALYTICAL FLOPs
# =============================================================================

print("\n[4/8] Preparing FLOPs calculation")


def count_flops(model, input_shape=(1, 3, IMAGE_SIZE, IMAGE_SIZE)):
    """
    Analytical FLOPs estimate.

    Counts:
        Conv2D:
            output_elements *
            kernel_height *
            kernel_width *
            input_channels/groups *
            2

        Linear:
            batch_size *
            in_features *
            out_features *
            2

    Convention:
        One multiply-add = 2 FLOPs.

    This is appropriate for comparing these CNN architectures
    under the same input size.
    """

    flops = 0

    hooks = []

    def conv_hook(module, inputs, output):

        # output shape:
        # [batch, channels, height, width]

        batch_size = output.shape[0]
        out_channels = output.shape[1]
        out_h = output.shape[2]
        out_w = output.shape[3]

        kernel_h, kernel_w = module.kernel_size

        in_channels = module.in_channels
        groups = module.groups

        operations_per_output = (
            kernel_h
            * kernel_w
            * (in_channels // groups)
        )

        layer_flops = (
            batch_size
            * out_channels
            * out_h
            * out_w
            * operations_per_output
            * 2
        )

        nonlocal flops
        flops += layer_flops

    def linear_hook(module, inputs, output):

        batch_size = output.shape[0]

        layer_flops = (
            batch_size
            * module.in_features
            * module.out_features
            * 2
        )

        nonlocal flops
        flops += layer_flops

    for module in model.modules():

        if isinstance(module, nn.Conv2d):
            hooks.append(
                module.register_forward_hook(conv_hook)
            )

        elif isinstance(module, nn.Linear):
            hooks.append(
                module.register_forward_hook(linear_hook)
            )

    device = next(model.parameters()).device

    dummy = torch.randn(
        input_shape,
        device=device,
        dtype=torch.float32
    )

    model.eval()

    with torch.no_grad():
        _ = model(dummy)

    for hook in hooks:
        hook.remove()

    return flops


# =============================================================================
# 5. MODEL SIZE / PARAMETER COUNT
# =============================================================================

print("\n[5/8] Preparing model-size measurement")


def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


def serialized_model_size_mb(model):
    """
    Measures serialized FP32 state_dict size in MB.
    This is more realistic than simply multiplying parameter
    count by 4 because serialization metadata is included.
    """

    buffer = io.BytesIO()

    torch.save(
        model.state_dict(),
        buffer
    )

    size_bytes = buffer.getbuffer().nbytes

    return (
        size_bytes,
        size_bytes / (1024 ** 2)
    )


# =============================================================================
# 6. CPU LATENCY
# =============================================================================

print("\n[6/8] CPU benchmarking")


def benchmark_cpu(model):

    torch.set_num_threads(CPU_THREADS)

    model = model.cpu()
    model.eval()

    dummy = torch.randn(
        BATCH_SIZE,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        dtype=torch.float32
    )

    # Warm-up
    with torch.no_grad():

        for _ in range(WARMUP_ITERS):
            _ = model(dummy)

    # Measurement
    timings_ms = []

    with torch.no_grad():

        for _ in range(MEASURE_ITERS):

            start = time.perf_counter()

            _ = model(dummy)

            end = time.perf_counter()

            timings_ms.append(
                (end - start) * 1000.0
            )

    timings_ms = torch.tensor(
        timings_ms,
        dtype=torch.float64
    )

    mean_ms = timings_ms.mean().item()
    median_ms = timings_ms.median().item()

    p95_ms = torch.quantile(
        timings_ms,
        0.95
    ).item()

    fps = (
        1000.0 / mean_ms
        if mean_ms > 0
        else float("nan")
    )

    return {
        "cpu_latency_mean_ms": mean_ms,
        "cpu_latency_median_ms": median_ms,
        "cpu_latency_p95_ms": p95_ms,
        "cpu_fps": fps,
    }


# =============================================================================
# 7. GPU LATENCY + MEMORY
# =============================================================================

print("\n[7/8] GPU benchmarking")


def benchmark_gpu(model):

    if not CUDA_AVAILABLE:

        return {
            "gpu_latency_mean_ms": None,
            "gpu_latency_median_ms": None,
            "gpu_latency_p95_ms": None,
            "gpu_fps": None,
            "gpu_peak_memory_mb": None,
            "gpu_available": False,
        }

    device = torch.device("cuda")

    torch.cuda.empty_cache()

    model = model.to(device)
    model.eval()

    dummy = torch.randn(
        BATCH_SIZE,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        device=device,
        dtype=torch.float32
    )

    # Reset peak statistics before the complete benchmark.
    torch.cuda.reset_peak_memory_stats(device)

    # -------------------------------------------------------------------------
    # Warm-up
    # -------------------------------------------------------------------------

    with torch.no_grad():

        for _ in range(WARMUP_ITERS):
            _ = model(dummy)

    torch.cuda.synchronize()

    # Reset after warm-up so the reported memory reflects
    # steady-state inference rather than warm-up artifacts.
    torch.cuda.reset_peak_memory_stats(device)

    # -------------------------------------------------------------------------
    # CUDA event latency
    # -------------------------------------------------------------------------

    timings_ms = []

    with torch.no_grad():

        for _ in range(MEASURE_ITERS):

            start_event = torch.cuda.Event(
                enable_timing=True
            )

            end_event = torch.cuda.Event(
                enable_timing=True
            )

            start_event.record()

            _ = model(dummy)

            end_event.record()

            torch.cuda.synchronize()

            elapsed = start_event.elapsed_time(
                end_event
            )

            timings_ms.append(elapsed)

    timings_ms = torch.tensor(
        timings_ms,
        dtype=torch.float64
    )

    mean_ms = timings_ms.mean().item()
    median_ms = timings_ms.median().item()

    p95_ms = torch.quantile(
        timings_ms,
        0.95
    ).item()

    fps = (
        1000.0 / mean_ms
        if mean_ms > 0
        else float("nan")
    )

    peak_memory_bytes = (
        torch.cuda.max_memory_allocated(device)
    )

    peak_memory_mb = (
        peak_memory_bytes
        / (1024 ** 2)
    )

    return {
        "gpu_latency_mean_ms": mean_ms,
        "gpu_latency_median_ms": median_ms,
        "gpu_latency_p95_ms": p95_ms,
        "gpu_fps": fps,
        "gpu_peak_memory_mb": peak_memory_mb,
        "gpu_available": True,
    }


# =============================================================================
# 8. RUN BENCHMARK
# =============================================================================

print("\n[8/8] Running benchmark")
print("-" * 80)

results = []

for model_name, builder in MODEL_BUILDERS.items():

    print(f"\n>>> {model_name}")

    # -------------------------------------------------------------------------
    # Build fresh model
    # -------------------------------------------------------------------------

    model = builder()

    model.eval()

    # -------------------------------------------------------------------------
    # Parameters
    # -------------------------------------------------------------------------

    params = count_parameters(model)

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    # -------------------------------------------------------------------------
    # Model serialized size
    # -------------------------------------------------------------------------

    size_bytes, size_mb = serialized_model_size_mb(
        model
    )

    # -------------------------------------------------------------------------
    # FLOPs
    # -------------------------------------------------------------------------

    model = model.cpu()

    flops = count_flops(
        model,
        input_shape=(
            BATCH_SIZE,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    )

    flops_g = flops / 1e9

    # -------------------------------------------------------------------------
    # CPU
    # -------------------------------------------------------------------------

    cpu_result = benchmark_cpu(model)

    # -------------------------------------------------------------------------
    # GPU
    # -------------------------------------------------------------------------

    gpu_result = benchmark_gpu(model)

    # -------------------------------------------------------------------------
    # Cleanup
    # -------------------------------------------------------------------------

    del model

    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # Store
    # -------------------------------------------------------------------------

    row = {
        "model": model_name,

        "input_size": "224x224",
        "batch_size": BATCH_SIZE,
        "num_classes": NUM_CLASSES,
        "precision": "FP32",

        "parameters": params,
        "trainable_parameters": trainable_params,

        "model_size_bytes": size_bytes,
        "model_size_mb": size_mb,

        "flops": flops,
        "flops_g": flops_g,

        **cpu_result,
        **gpu_result,
    }

    results.append(row)

    # -------------------------------------------------------------------------
    # Print
    # -------------------------------------------------------------------------

    print(
        f"Parameters       : {params:,}"
    )

    print(
        f"Model size       : {size_mb:.2f} MB"
    )

    print(
        f"FLOPs            : {flops_g:.3f} GFLOPs"
    )

    print(
        f"CPU latency      : "
        f"{cpu_result['cpu_latency_mean_ms']:.3f} ms"
    )

    print(
        f"CPU FPS          : "
        f"{cpu_result['cpu_fps']:.2f}"
    )

    if CUDA_AVAILABLE:

        print(
            f"GPU latency      : "
            f"{gpu_result['gpu_latency_mean_ms']:.3f} ms"
        )

        print(
            f"GPU FPS          : "
            f"{gpu_result['gpu_fps']:.2f}"
        )

        print(
            f"Peak GPU memory  : "
            f"{gpu_result['gpu_peak_memory_mb']:.2f} MB"
        )

    else:

        print("GPU latency      : N/A")
        print("GPU FPS          : N/A")
        print("Peak GPU memory  : N/A")


# =============================================================================
# 9. RESULTS TABLE
# =============================================================================

results_df = pd.DataFrame(results)

# Reorder columns for readability
column_order = [
    "model",
    "input_size",
    "batch_size",
    "num_classes",
    "precision",

    "parameters",
    "trainable_parameters",

    "model_size_mb",

    "flops_g",

    "cpu_latency_mean_ms",
    "cpu_latency_median_ms",
    "cpu_latency_p95_ms",
    "cpu_fps",

    "gpu_latency_mean_ms",
    "gpu_latency_median_ms",
    "gpu_latency_p95_ms",
    "gpu_fps",

    "gpu_peak_memory_mb",
    "gpu_available",
]

results_df = results_df[column_order]

print("\n" + "=" * 80)
print("STAGE 4.3 — COMPUTATIONAL BENCHMARK RESULTS")
print("=" * 80)

display(results_df)


# =============================================================================
# 10. RANKING SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("EFFICIENCY RANKINGS")
print("=" * 80)

ranking = results_df.copy()

ranking["parameter_rank"] = (
    ranking["parameters"]
    .rank(method="min")
)

ranking["flops_rank"] = (
    ranking["flops_g"]
    .rank(method="min")
)

ranking["cpu_latency_rank"] = (
    ranking["cpu_latency_mean_ms"]
    .rank(method="min")
)

if CUDA_AVAILABLE:

    ranking["gpu_latency_rank"] = (
        ranking["gpu_latency_mean_ms"]
        .rank(method="min")
    )

    ranking["gpu_memory_rank"] = (
        ranking["gpu_peak_memory_mb"]
        .rank(method="min")
    )

    ranking["gpu_fps_rank"] = (
        ranking["gpu_fps"]
        .rank(method="min", ascending=False)
    )

print("\nSmallest parameter count:")
print(
    ranking.loc[
        ranking["parameters"].idxmin(),
        "model"
    ]
)

print("\nLowest FLOPs:")
print(
    ranking.loc[
        ranking["flops_g"].idxmin(),
        "model"
    ]
)

print("\nLowest CPU latency:")
print(
    ranking.loc[
        ranking["cpu_latency_mean_ms"].idxmin(),
        "model"
    ]
)

if CUDA_AVAILABLE:

    print("\nLowest GPU latency:")
    print(
        ranking.loc[
            ranking["gpu_latency_mean_ms"].idxmin(),
            "model"
        ]
    )

    print("\nHighest GPU FPS:")
    print(
        ranking.loc[
            ranking["gpu_fps"].idxmax(),
            "model"
        ]
    )

    print("\nLowest peak GPU memory:")
    print(
        ranking.loc[
            ranking["gpu_peak_memory_mb"].idxmin(),
            "model"
        ]
    )


# =============================================================================
# 11. SAVE CSV
# =============================================================================

csv_path = (
    OUTPUT_DIR
    / "stage4_3_computational_benchmark.csv"
)

results_df.to_csv(
    csv_path,
    index=False
)


# =============================================================================
# 12. SAVE PROTOCOL JSON
# =============================================================================

protocol = {
    "stage": "4.3",
    "title": "Computational Benchmarking",

    "purpose": (
        "Fair computational comparison of ResNet18, "
        "MobileNetV3-Large, and EfficientNet-B0 "
        "for the CISLR frame-classification architecture."
    ),

    "candidate_models": [
        "ResNet18",
        "MobileNetV3-Large",
        "EfficientNet-B0",
    ],

    "canonical_pool": {
        "videos": 4764,
        "source": (
            "stage4_2_canonical_training_manifest.csv"
        ),
        "official_test_used": False,
        "new_validation_split_created": False,
    },

    "benchmark": {
        "input_shape": [
            1,
            3,
            224,
            224
        ],
        "batch_size": 1,
        "num_classes": NUM_CLASSES,
        "precision": "FP32",

        "warmup_iterations": WARMUP_ITERS,
        "measurement_iterations": MEASURE_ITERS,

        "cpu_threads": CPU_THREADS,

        "flops_convention": (
            "1 multiply-add = 2 FLOPs"
        ),

        "latency_measurement": {
            "cpu": "time.perf_counter",
            "gpu": "CUDA events",
        },
    },

    "scientific_constraints": [
        "No model training performed.",
        "No official test videos used.",
        "No new validation split created.",
        "All candidate models use the same input size.",
        "All candidate models use batch size 1.",
        "All candidate models use FP32.",
        "All candidate models use the same 4764-class classifier output.",
        "CPU benchmarking uses one thread for deterministic comparison.",
    ],

    "hardware": {
        "platform": platform.platform(),
        "pytorch": torch.__version__,
        "torchvision": torchvision.__version__,
        "cuda_available": CUDA_AVAILABLE,
        "gpu_name": GPU_NAME,
        "cuda_version": CUDA_VERSION,
    },

    "results_csv": str(csv_path),
}


json_path = (
    OUTPUT_DIR
    / "stage4_3_computational_benchmark.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "results": results_df.to_dict(
                orient="records"
            ),
            "protocol": protocol,
        },
        f,
        indent=2
    )


# =============================================================================
# 13. HASH ARTIFACTS
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


protocol_path = (
    OUTPUT_DIR
    / "stage4_3_protocol.json"
)

with open(
    protocol_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )


# =============================================================================
# 14. FINAL REPORT
# =============================================================================

print("\n" + "=" * 80)
print("STAGE 4.3 COMPLETE")
print("=" * 80)

print("\nArtifacts:")

print(
    f"CSV      : {csv_path}"
)

print(
    f"JSON     : {json_path}"
)

print(
    f"Protocol : {protocol_path}"
)

print("\nSHA-256:")

print(
    f"CSV      : {sha256_file(csv_path)}"
)

print(
    f"JSON     : {sha256_file(json_path)}"
)

print(
    f"Protocol : {sha256_file(protocol_path)}"
)

print("\nScientific status:")
print("✓ No training performed")
print("✓ No official test data used")
print("✓ No new validation split created")
print("✓ Same 224x224 input")
print("✓ Same batch size = 1")
print("✓ Same FP32 precision")
print("✓ Same 4764-class classifier head")
print("✓ CPU and GPU latency measured under fixed conditions")

print("\nNext stage:")
print(
    "Stage 4.4/4.5 — Train the three candidate backbones "
    "under the frozen Stage 4.2 protocol and collect "
    "supervised training diagnostics."
)

print("=" * 80)

CISLR — STAGE 4.3 COMPUTATIONAL BENCHMARKING

[1/8] Environment
PyTorch       : 2.14.0+cu130
Torchvision   : 0.29.0+cu130
Python        : 3.11.15
Platform      : Linux-7.0.0-30-generic-x86_64-with-glibc2.39
CUDA available: True
GPU           : NVIDIA GeForce RTX 4050 Laptop GPU
CUDA          : 13.0
CPU threads   : 1

[2/8] Verifying Stage 4.2 canonical training pool
Manifest rows : 38,112
Videos        : 4,764
✓ Stage 4.2 canonical pool verified: 4764 videos

[3/8] Building candidate architectures

[4/8] Preparing FLOPs calculation

[5/8] Preparing model-size measurement

[6/8] CPU benchmarking

[7/8] GPU benchmarking

[8/8] Running benchmark
--------------------------------------------------------------------------------

>>> ResNet18
Parameters       : 13,620,444
Model size       : 52.03 MB
FLOPs            : 3.632 GFLOPs
CPU latency      : 59.379 ms
CPU FPS          : 16.84
GPU latency      : 2.391 ms
GPU FPS          : 418.24
Peak GPU memory  : 71.28 MB

>>> MobileNetV3-Large
Downl

100%|██████████| 21.1M/21.1M [00:10<00:00, 2.05MB/s]


Parameters       : 10,304,716
Model size       : 39.50 MB
FLOPs            : 0.443 GFLOPs
CPU latency      : 18.989 ms
CPU FPS          : 52.66
GPU latency      : 5.362 ms
GPU FPS          : 186.51
Peak GPU memory  : 56.62 MB

>>> EfficientNet-B0
Parameters       : 10,110,232
Model size       : 38.85 MB
FLOPs            : 0.781 GFLOPs
CPU latency      : 34.335 ms
CPU FPS          : 29.12
GPU latency      : 7.563 ms
GPU FPS          : 132.23
Peak GPU memory  : 59.10 MB

STAGE 4.3 — COMPUTATIONAL BENCHMARK RESULTS


,model,input_size,batch_size,num_classes,precision,parameters,trainable_parameters,model_size_mb,flops_g,cpu_latency_mean_ms,cpu_latency_median_ms,cpu_latency_p95_ms,cpu_fps,gpu_latency_mean_ms,gpu_latency_median_ms,gpu_latency_p95_ms,gpu_fps,gpu_peak_memory_mb,gpu_available
0,ResNet18,224x224,1,4764,FP32,13620444,13620444,52.031260,3.632001,59.378838,59.710903,63.024362,16.841017,2.390984,2.359520,2.880717,418.237874,71.284180,True
1,MobileNetV3-Large,224x224,1,4764,FP32,10304716,10304716,39.500625,0.442815,18.988631,18.979570,21.177363,52.663090,5.361543,5.321728,6.235958,186.513460,56.620605,True
2,EfficientNet-B0,224x224,1,4764,FP32,10110232,10110232,38.846725,0.781265,34.335366,34.428725,37.112767,29.124490,7.562565,7.383936,9.565850,132.230275,59.101074,True



EFFICIENCY RANKINGS

Smallest parameter count:
EfficientNet-B0

Lowest FLOPs:
MobileNetV3-Large

Lowest CPU latency:
MobileNetV3-Large

Lowest GPU latency:
ResNet18

Highest GPU FPS:
ResNet18

Lowest peak GPU memory:
MobileNetV3-Large

STAGE 4.3 COMPLETE

Artifacts:
CSV      : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_3_benchmark/stage4_3_computational_benchmark.csv
JSON     : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_3_benchmark/stage4_3_computational_benchmark.json
Protocol : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_3_benchmark/stage4_3_protocol.json

SHA-256:
CSV      : 813af713af065d5a9f06a85279d56c8ff3158b1a68fea3f3b4cb3ac6ecd45a51
JSON     : 77f2f7c0ade0a41dc8620320e09d9df6fe9656a25103238a98a18873cb78f961
Protocol : c858d49d382ccf3d249ac2da25e26991fd5411bc00c48f91f7fbd93bc7156f4c

Scientific status:
✓ No training performed
✓ No official test data used
✓ No new validation split created
✓ Same 224x224 input
✓ Same batch size = 1
✓ Same FP32 pre